# 🏠 House Price Prediction with Neural Networks — Improved 

This is the **second part** of our lecture. In the previous notebook we built a complete, working neural network pipeline. It works, but can we do better?

In this notebook we keep the **same structure** and apply **5 targeted tricks** to improve performance:

| # | Trick | What it does |
|---|-------|-------------|
| 🔧 1 | **Log-transform del target** | The model learns *relative* errors instead of absolute dollar errors |
| 🔧 2 | **Feature engineering** | We give the network pre-computed interactions that are hard to learn from raw features |
| 🔧 3 | **Mini-batch training (DataLoader)** | Faster convergence and better generalization with stochastic gradient estimates |
| 🔧 4 | **Learning Rate Scheduler** | Automatically reduces the learning rate when training plateaus |
| 🔧 5 | **Kaiming weight initialization** | Smarter starting weights matched to ReLU activations |

Let's see how each trick contributes to better results.

---

## 📦 Step 0: Install Required Packages

Same packages as the base notebook.

In [ ]:
# ============================================
# INSTALLATION — Run this first!
# ============================================

%pip install torch scikit-learn kagglehub pandas numpy matplotlib seaborn --quiet

print("✅ All packages installed successfully!")

## 📦 Step 1: Import Libraries

Same imports as the base notebook, with one addition: `torch.utils.data` for the **DataLoader** (Trick 3).

In [ ]:
# ============================================
# IMPORT LIBRARIES
# ============================================

# --- Deep learning ---
import torch                             # Core PyTorch library
import torch.nn as nn                    # Neural network building blocks
from torch.utils.data import TensorDataset, DataLoader  # 🔧 NEW: Mini-batch training

# --- Data handling ---
import numpy as np                       # Numerical arrays and math
import pandas as pd                      # DataFrames (tabular data)
import kagglehub                         # Download datasets from Kaggle
import os                                # File-system utilities

# --- Machine learning utilities ---
from sklearn.model_selection import train_test_split  # Split data
from sklearn.preprocessing import StandardScaler      # Scale data

# --- Visualization ---
import matplotlib.pyplot as plt          # Static plots
import seaborn as sns                    # Statistical plots

# Apply a clean visual style for all plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ All libraries imported successfully!")

---

# 📊 1. Data Loading & Exploration

This section is **identical** to the base notebook — same dataset, same exploration.

In [ ]:
# ============================================
# DOWNLOAD DATASET FROM KAGGLE
# ============================================
# kagglehub.dataset_download() fetches the dataset
# and caches it locally so it won't re-download next time.

print("📥 Downloading dataset from Kaggle...")
dataset_path = kagglehub.dataset_download(
    'muhamedumarjamil/house-price-prediction-dataset'
)
print(f"✅ Dataset downloaded to: {dataset_path}")

In [ ]:
# ============================================
# LOAD CSV INTO A PANDAS DATAFRAME
# ============================================

csv_file = os.path.join(dataset_path, 'house_prices_dataset.csv')
df = pd.read_csv(csv_file)

print(f"✅ Dataset loaded!")
print(f"   Rows (houses):    {len(df)}")
print(f"   Columns (features): {len(df.columns)}")
print(f"   Column names: {list(df.columns)}")

In [ ]:
# ============================================
# DATA CLEANING
# ============================================

print(f"📊 Original dataset size: {len(df)} samples")

# Remove extreme outliers using log-space (±3 std)
# This helps the model learn better by removing houses with unusual prices
log_prices = np.log(df['price'])
mean_log = log_prices.mean()
std_log = log_prices.std()
outlier_mask = np.abs(log_prices - mean_log) <= 3 * std_log
df = df[outlier_mask].copy()

print(f"   After removing outliers: {len(df)} samples")
print(f"✅ Data cleaning complete!")

In [ ]:
# ============================================
# DATA VISUALIZATION
# ============================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- 1. Distribution of house prices ---
axes[0, 0].hist(df['price'], bins=50, color='steelblue',
                edgecolor='white', alpha=0.7)
axes[0, 0].set_xlabel('House Price ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('📊 Distribution of House Prices')
axes[0, 0].axvline(df['price'].mean(), color='red', linestyle='--',
                    label=f'Mean: ${df["price"].mean():,.0f}')
axes[0, 0].legend()

# --- 2. Feature correlation with price ---
feature_cols = [c for c in df.columns if c != 'price']
correlations = [df[c].corr(df['price']) for c in feature_cols]
colors = ['green' if c > 0 else 'red' for c in correlations]
axes[0, 1].barh(feature_cols, correlations, color=colors, alpha=0.7)
axes[0, 1].set_xlabel('Correlation with Price')
axes[0, 1].set_title('📈 Feature Correlation with House Price')
axes[0, 1].axvline(0, color='black', linewidth=0.5)

# --- 3. Square Feet vs Price ---
axes[1, 0].scatter(df['square_feet'], df['price'], alpha=0.3, s=5,
                   c='steelblue')
axes[1, 0].set_xlabel('Square Feet')
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].set_title('📐 House Size vs Price')

# --- 4. Age vs Price ---
axes[1, 1].scatter(df['age'], df['price'], alpha=0.3, s=5,
                   c='green')
axes[1, 1].set_xlabel('Age (years)')
axes[1, 1].set_ylabel('Price ($)')
axes[1, 1].set_title('🕰️ House Age vs Price')

plt.tight_layout()
plt.show()

---

# 🔧 2. Data Preparation — With Tricks

This is where the first two tricks come in. Compare with the base notebook:

| Step | Base Notebook | This Notebook |
|------|--------------|---------------|
| Target | Raw price in $ → StandardScaler | 🔧 **Log-transform** of price |
| Features | 4 original features | 🔧 **8 features** (4 original + 4 engineered) |
| Tensors | Plain tensors | 🔧 Wrapped in **DataLoader** for mini-batches |

## 🔧 Trick 1: Log-Transform the Target

### The problem with raw prices

In the base notebook we predicted the raw price in dollars and used a StandardScaler on the target. This has a subtle issue: the model optimizes **absolute dollar errors** — an error of \$10,000 counts the same whether the house costs \$50,000 or \$500,000. But in real estate, a \$10k error on a \$50k house (20%) is much worse than on a \$500k house (2%).

### The fix

By applying `log()` to the prices before training, we transform the problem:

$$\text{loss} = (\log(\hat{y}) - \log(y))^2 \approx \left(\frac{\hat{y} - y}{y}\right)^2$$

The model now optimizes **relative (percentage) errors**, which is exactly what we want for prices. To get back to dollars, we simply apply `exp()` to the predictions.

### Why it works here

Look at the price distribution from the plots above — it's right-skewed. The log transform:
- **Compresses** the range (large prices become less extreme)
- **Symmetrizes** the distribution (closer to a Gaussian)
- Makes MSE loss behave like **MAPE** (percentage error)

## 🔧 Trick 2: Feature Engineering

### The problem

A neural network with ReLU activations can theoretically approximate any function, but **multiplications between inputs are expensive to learn**. To approximate something as simple as $x_1 \times x_2$, a shallow network needs many neurons and many epochs.

### The fix

We create 4 new features with clear physical/economic meaning:

| New Feature | Formula | Why it matters |
|------------|---------|---------------|
| `room_size` | `square_feet / num_rooms` | Average room size — a proxy for "luxury". Same area with fewer rooms → bigger rooms → higher price |
| `total_space` | `square_feet × num_rooms` | Combined effect: many rooms AND large area → disproportionately expensive |
| `sqft_squared` | `square_feet²` | Price per sqft is not constant — larger houses have a superlinear price increase |
| `dist_squared` | `distance_to_city²` | Distance effect saturates: 1km→5km matters a lot, 40km→44km barely matters |

> ⚠️ **Why not more features?** Adding too many features (e.g., `PolynomialFeatures(degree=3)` on everything) risks introducing noise and multicollinearity. We pick **few, motivated features** based on domain knowledge.

In [ ]:
# ============================================
# 🔧 TRICK 2: FEATURE ENGINEERING
# ============================================

# TODO: Create new features with physical/economic meaning
df['room_size']    = df['square_feet'] / df['num_rooms']       # Average sqft per room
df['total_space']  =                                           # Total usable space
df['sqft_squared'] =                                           # Superlinear price effect
df['dist_squared'] =                                           # Saturating distance effect

print(f"✅ Feature engineering complete!")
print(f"   Original features:    4")
print(f"   Engineered features:  4")
print(f"   Total features:       {len(df.columns) - 1}")  # -1 for 'price'
print(f"\n   New columns: {['room_size', 'total_space', 'sqft_squared', 'dist_squared']}")

### 2a. Separate features and target (with log-transform)

In [ ]:
# ============================================
# SEPARATE FEATURES AND TARGET (with log-transform)
# ============================================

# Feature columns — everything except 'price'
feature_columns = [c for c in df.columns if c != 'price']

features = df[feature_columns].values                        # shape: (N, 8) — now 8 features!

#TODO: compose the log_prices array here by applying np.log to the 'price' column of the dataframe and reshaping it to (-1, 1)
# 🔧 TRICK 1: Log-transform the prices
log_prices =                   # shape: (N, 1)

# Keep original prices for final evaluation (to compute metrics in dollars)
original_prices = df['price'].values.reshape(-1, 1)

print(f"Number of samples:  {len(log_prices)}")
print(f"Number of features: {features.shape[1]}")
print(f"Feature names:      {feature_columns}")
print(f"\n📊 Price statistics:")
print(f"   Original range:  ${original_prices.min():,.0f} - ${original_prices.max():,.0f}")
print(f"   Log price range: {log_prices.min():.2f} - {log_prices.max():.2f}")

### 2b. Split the data

Identical split strategy: 80/10/10 with `random_state=42`.

In [ ]:
# ============================================
# SPLIT THE DATA INTO TRAIN / VAL / TEST
# ============================================

# First split: 80% training, 20% temporary
(train_features, temp_features,
 train_log_prices, temp_log_prices,
 train_orig_prices, temp_orig_prices) = train_test_split(
    features, log_prices, original_prices,
    test_size=0.2, random_state=42
)

# Second split: 50/50 on the temporary set → 10% val + 10% test
(val_features, test_features,
 val_log_prices, test_log_prices,
 val_orig_prices, test_orig_prices) = train_test_split(
    temp_features, temp_log_prices, temp_orig_prices,
    test_size=0.5, random_state=42
)

print(f"Training samples:   {len(train_log_prices)}")
print(f"Validation samples: {len(val_log_prices)}")
print(f"Test samples:       {len(test_log_prices)}")

### 2c. Scale the features

Same StandardScaler approach. Note that we only scale **features**, not the log-prices (they're already in a reasonable range).

In [ ]:
# ============================================
# SCALE THE DATA
# ============================================

# Scaler for features ONLY
feature_scaler = StandardScaler()
train_features = feature_scaler.fit_transform(train_features)
val_features   = feature_scaler.transform(val_features)
test_features  = feature_scaler.transform(test_features)

# Log prices are NOT scaled — they remain as-is
# This makes it easier to convert predictions back to real prices with np.exp()

print("✅ Features scaled — all feature values now have mean ≈ 0 and std ≈ 1.")
print("   Log prices are NOT scaled (will use np.exp() to convert back)")

### 2d. Convert to PyTorch tensors + DataLoader

## 🔧 Trick 3: Mini-Batch Training with DataLoader

### The problem with full-batch

In the base notebook, every epoch processes the **entire training set at once**. This means:
- The gradient is computed on all ~8000 samples — very stable but slow to converge
- The model sees the same "big picture" every step — no exploration of the loss landscape
- Memory usage scales linearly with dataset size (problematic for large datasets)

### The fix

We wrap our tensors in a `DataLoader` with `batch_size=256`. Each epoch now processes the data in **mini-batches** of 256 samples:

```
Epoch 1:  batch_1 (256) → update → batch_2 (256) → update → ... → batch_31 (256) → update
Epoch 2:  [shuffled] batch_1 → update → batch_2 → update → ...
```

### Why it works
- **Noisy gradients act as regularization** — the model doesn't overfit to the full-batch gradient direction
- **More weight updates per epoch** — ~31 updates instead of 1
- **Shuffling** ensures different mini-batches each epoch, exposing the model to varied data orderings
- **Scales to any dataset size** — memory usage depends on batch size, not dataset size

In [ ]:
# ============================================
# CONVERT TO PYTORCH TENSORS + DATALOADER
# ============================================

# Convert features to tensors
train_features_t = torch.tensor(train_features, dtype=torch.float32)
val_features_t   = torch.tensor(val_features,   dtype=torch.float32)
test_features_t  = torch.tensor(test_features,  dtype=torch.float32)

# Convert log prices to tensors
train_log_prices_t = torch.tensor(train_log_prices, dtype=torch.float32)
val_log_prices_t   = torch.tensor(val_log_prices,   dtype=torch.float32)
test_log_prices_t  = torch.tensor(test_log_prices,  dtype=torch.float32)

# 🔧 TRICK 3: Create DataLoader for mini-batch training
#TODO: Create a TensorDataset from the training features and log prices, 
#Then create a DataLoader with batch_size=256 and shuffle=True for training.
train_dataset = 
train_loader  = 

print(f"Training tensor shapes:   features {train_features_t.shape}, log_prices {train_log_prices_t.shape}")
print(f"Validation tensor shapes: features {val_features_t.shape}, log_prices {val_log_prices_t.shape}")
print(f"Test tensor shapes:       features {test_features_t.shape}, log_prices {test_log_prices_t.shape}")
print(f"\n🔧 DataLoader: {len(train_loader)} batches of 256 samples per epoch")

---

# 🧠 3. Neural Network Architecture

Same architecture as the base notebook (`128 → 128 → 64 → 1`), but with two changes:
- **8 input features** instead of 4 (due to feature engineering)
- **Kaiming weight initialization** (Trick 5)

## 🔧 Trick 5: Kaiming (He) Weight Initialization

### The problem with default initialization

PyTorch initializes weights using a uniform distribution that doesn't account for the activation function. With ReLU, which zeros out all negative values, this can lead to:
- **Dead neurons** — some neurons always output 0 and never recover
- **Vanishing/exploding activations** — values shrink or grow as they pass through layers

### The fix

**Kaiming initialization** (He et al., 2015) sets the weight variance to:

$$\text{Var}(w) = \frac{2}{n_{\text{in}}}$$

where $n_{\text{in}}$ is the number of inputs to the layer. The factor of 2 specifically compensates for ReLU zeroing out half the activations. This keeps the variance of activations roughly constant across layers.

### In practice

We add a simple `_init_weights()` method that applies Kaiming to all `Linear` layers and sets biases to zero:

```python
nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
nn.init.zeros_(layer.bias)
```

In [ ]:
# ============================================
# DEFINE THE NEURAL NETWORK MODEL (with Kaiming init)
# ============================================

class HousePriceModel(nn.Module):
    """
    A neural network for house price prediction.

    Architecture:
        Input → 128 → 128 → 64 → 1
        with BatchNorm, ReLU activations, Dropout, and Kaiming initialization.
    """

    def __init__(self, n_features, dropout_rate=0.2):
        super().__init__()

        self.network = nn.Sequential(
            # Layer 1: input features → 128 neurons
            nn.Linear(n_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Layer 2: 128 → 128 neurons
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Layer 3: 128 → 64 neurons
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate / 2),  # Reduced dropout near output

            # Output layer: 64 → 1 (predicted log price)
            nn.Linear(64, 1)
        )

        # 🔧 TRICK 5: Apply Kaiming initialization
        self._init_weights()

    def _init_weights(self):
        """Apply Kaiming (He) initialization to all Linear layers."""
        for module in self.network:
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
                nn.init.zeros_(module.bias)

    def forward(self, x):
        """Forward pass: push input x through all layers."""
        return self.network(x)


# Instantiate the model
model = HousePriceModel(
    n_features=train_features_t.shape[1],  # 8 features (4 original + 4 engineered)
    dropout_rate=0.2
)

print(model)
print(f"\n📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   (vs base notebook: input layer is now 8→128 instead of 4→128)")

---

# 🏋️ 4. Training the Model — With Tricks

Two more changes compared to the base notebook:
- **Mini-batch training** — iterate over the DataLoader instead of the full dataset
- **Learning Rate Scheduler** — automatically reduce LR when the model plateaus

## 🔧 Trick 4: Learning Rate Scheduler (ReduceLROnPlateau)

### The problem with a fixed learning rate

A fixed LR is a compromise:
- **Too high** → the model oscillates around the minimum and never converges precisely
- **Too low** → training is painfully slow

With a fixed LR, you're stuck with one speed for the entire journey.

### The fix

`ReduceLROnPlateau` monitors the validation loss. When it stops improving for `patience` epochs, the scheduler **divides the LR by a factor** (we use 0.5):

```
Early training:  LR = 0.001    (big steps, fast progress)
After plateau:   LR = 0.0005   (smaller steps, fine-tuning)
Finally:   LR = 0.00025  (even smaller, precision)
...
```

This gives us the best of both worlds: fast exploration early on, precise convergence later.

### Hyperparameters

| Hyperparameter | Value | Meaning |
|---------------|-------|---------|
| `learning_rate` | 0.001 | Initial step size |
| `weight_decay` | 1e-4 | L2 regularization |
| `dropout_rate` | 0.2 | Fraction of neurons disabled |
| `max_epochs` | 2000 | Maximum training loops |
| `batch_size` | 256 | 🔧 Samples per mini-batch |
| `scheduler factor` | 0.5 | 🔧 LR multiplier on plateau |
| `scheduler patience` | 50 | 🔧 Epochs to wait before reducing LR |

In [ ]:
# ============================================
# SET UP LOSS, OPTIMIZER, AND SCHEDULER
# ============================================

# MSELoss on log-prices
loss_function = nn.MSELoss()

# Learning rate and weight decay (same as base)
learning_rate = 0.001
weight_decay = 1e-4

# Adam optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

# 🔧 TRICK 4: Learning rate scheduler
#TODO: Complete the function call to create the ReduceLROnPlateau scheduler with the specified parameters,
# using the 'optimizer' variable defined above, and the following settings:
#    mode='min'       # Minimize validation loss
#    factor=0.5       # Reduce LR by half
#    patience=50      # Wait 50 epochs before reducing
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=,
    mode=       # Minimize validation loss
    factor=       # Reduce LR by half
    patience=       # Wait 50 epochs before reducing
)

number_of_epochs = 2000

print(f"Loss function: {loss_function}")
print(f"Optimizer:     Adam (lr={learning_rate}, weight_decay={weight_decay})")
print(f"🔧 Scheduler:  ReduceLROnPlateau (factor=0.5, patience=50)")
print(f"Max epochs:    {number_of_epochs}")

### The training loop (mini-batch + scheduler)

Key differences from the base notebook (highlighted with 🔧):
- **Inner loop** over `train_loader` batches instead of one full-batch forward pass
- **Scheduler step** after each epoch based on validation loss
- **Learning rate logging** to see when the scheduler kicks in

In [ ]:
# ============================================
# TRAINING LOOP (Mini-batch + Scheduler + Early Stopping)
# ============================================

train_losses = []
val_losses   = []

# Early stopping setup
best_val_loss = float('inf')
patience_counter = 0
patience = 100
best_model_state = None

for epoch in range(number_of_epochs):

    # --- TRAINING PHASE (mini-batch) ---
    model.train()
    epoch_train_loss = 0.0

    # 🔧 TRICK 3: Iterate over mini-batches
    for batch_features, batch_prices in train_loader:
        predictions = model(batch_features)                  # Forward pass on batch
        batch_loss = loss_function(predictions, batch_prices) # Loss on batch

        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        epoch_train_loss += batch_loss.item() * len(batch_features)

    # Average training loss across all samples
    epoch_train_loss /= len(train_loader.dataset)

    # --- VALIDATION PHASE (full validation set, no batching needed) ---
    model.eval()
    with torch.no_grad():
        val_predictions = model(val_features_t)
        val_loss = loss_function(val_predictions, val_log_prices_t)

    # 🔧 TRICK 4: Update learning rate scheduler
    scheduler.step(val_loss)

    # Store losses
    train_losses.append(epoch_train_loss)
    val_losses.append(val_loss.item())

    # --- EARLY STOPPING CHECK ---
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        patience_counter = 0
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"\n⏹️  Early stopping triggered at epoch {epoch}")
        print(f"   No improvement for {patience} epochs")
        break

    # Print progress every 100 epochs
    if epoch % 100 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(
            f"Epoch {epoch:>4d} | "
            f"Train: {epoch_train_loss:.4f} | "
            f"Val: {val_loss.item():.4f} | "
            f"LR: {current_lr:.6f}"
        )

# Load the best model state
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✅ Loaded best model from epoch with val_loss = {best_val_loss:.4f}")

print(f"\n✅ Training complete!")
print(f"   Total epochs run:      {epoch + 1}")
print(f"   Best validation loss:  {best_val_loss:.4f}")

### Visualizing the training history

In [ ]:
# ============================================
# TRAINING HISTORY VISUALIZATION
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full loss curves (log scale for better visibility)
axes[0].plot(train_losses, label='Training Loss', alpha=0.8)
axes[0].plot(val_losses,   label='Validation Loss', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss (log scale)')
axes[0].set_title('📉 Training & Validation Loss Over Time')
axes[0].legend()
axes[0].set_yscale('log')

# Right: zoomed view of last portion of training
zoom_epochs = min(200, len(train_losses))
axes[1].plot(range(len(train_losses) - zoom_epochs, len(train_losses)),
             train_losses[-zoom_epochs:], label='Training Loss', alpha=0.8)
axes[1].plot(range(len(val_losses) - zoom_epochs, len(val_losses)),
             val_losses[-zoom_epochs:], label='Validation Loss', alpha=0.8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].set_title(f'🔍 Loss Convergence (Last {zoom_epochs} Epochs)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n✅ Final Training Loss:   {train_losses[-1]:.4f}")
print(f"✅ Final Validation Loss: {val_losses[-1]:.4f}")

---

# 📏 5. Model Evaluation

Same metrics as the base notebook. The key difference is that our model outputs **log-prices**, so we apply `np.exp()` to convert predictions back to dollars before computing metrics.

Compare these numbers with the base notebook to see the impact of our 5 tricks!

In [ ]:
# ============================================
# GENERATE PREDICTIONS AND CONVERT FROM LOG SPACE
# ============================================

model.eval()
with torch.no_grad():
    # Get predictions in log space
    test_pred_log = model(test_features_t).numpy()

# 🔧 TRICK 1: Convert from log space back to real prices using exp()
test_predictions = np.exp(test_pred_log)

# Use original (non-log) test prices for evaluation
test_prices_original = test_orig_prices

In [ ]:
# ============================================
# OVERALL PERFORMANCE METRICS
# ============================================

# Flatten arrays for metrics calculation
actual = test_prices_original.flatten()
pred = test_predictions.flatten()

# Calculate overall metrics
mae = np.mean(np.abs(pred - actual))
rmse = np.sqrt(np.mean((pred - actual) ** 2))
mape = np.mean(np.abs((pred - actual) / actual)) * 100

ss_res = np.sum((actual - pred) ** 2)
ss_tot = np.sum((actual - np.mean(actual)) ** 2)
r2_score = 1 - (ss_res / ss_tot)

print("=" * 55)
print("           OVERALL TEST SET PERFORMANCE")
print("           (Improved with 5 Tricks)")
print("=" * 55)
print()
print(f"   MAE  (Mean Absolute Error):      ${mae:,.2f}")
print(f"   RMSE (Root Mean Squared Error):  ${rmse:,.2f}")
print(f"   MAPE (Mean Absolute % Error):    {mape:.2f}%")
print(f"   R²   (Coefficient of Determination): {r2_score:.4f}")
print()
print("=" * 55)
print()
print("📝 Compare these with the base notebook to see the improvement!")

In [ ]:
# ============================================
# PERFORMANCE BY PRICE RANGE
# ============================================

# Define quartile boundaries
quartiles = np.quantile(actual, [0, 0.25, 0.5, 0.75, 1.0])

print("\n" + "=" * 55)
print("        PERFORMANCE BY PRICE RANGE (QUARTILES)")
print("=" * 55)

for i in range(4):
    # Select samples in this price range
    mask = (actual >= quartiles[i]) & (actual < quartiles[i+1])
    a = actual[mask]
    p = pred[mask]

    # Metrics
    mae_q  = np.mean(np.abs(p - a))
    rmse_q = np.sqrt(np.mean((p - a) ** 2))
    mape_q = np.mean(np.abs((p - a) / a)) * 100

    print()
    print(f"Range {i+1}: ${quartiles[i]:,.0f} → ${quartiles[i+1]:,.0f}")
    print(f"   MAE:  ${mae_q:,.2f}")
    print(f"   RMSE: ${rmse_q:,.2f}")
    print(f"   MAPE: {mape_q:.2f}%")

print()
print("💡 Notice how the log-transform (Trick 1) makes MAPE more uniform across quartiles!")

---

# 📊 6. Prediction Visualizations

Same three plots as the base notebook. Look for tighter clustering around the diagonal and a narrower residual distribution.

In [ ]:
# ============================================
# PREDICTION VISUALIZATIONS
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- 1. Predicted vs Actual scatter ---
axes[0].scatter(test_prices_original, test_predictions,
                alpha=0.5, s=10)
lo = min(test_prices_original.min(), test_predictions.min())
hi = max(test_prices_original.max(), test_predictions.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='Perfect')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('🎯 Predicted vs Actual Prices')
axes[0].legend()

# --- 2. Residuals distribution ---
residuals = (test_predictions - test_prices_original).flatten()
axes[1].hist(residuals, bins=50, color='steelblue',
             edgecolor='white', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prediction Error ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('📊 Distribution of Prediction Errors')

# --- 3. Sample comparison bar chart ---
np.random.seed(42)
sample_idx = np.random.choice(len(test_prices_original), 20, replace=False)
x_pos = np.arange(20)
width = 0.35
axes[2].bar(x_pos - width / 2,
            test_prices_original[sample_idx].flatten(),
            width, label='Actual', alpha=0.8)
axes[2].bar(x_pos + width / 2,
            test_predictions[sample_idx].flatten(),
            width, label='Predicted', alpha=0.8)
axes[2].set_xlabel('Sample Index')
axes[2].set_ylabel('Price ($)')
axes[2].set_title('🏠 Sample Predictions vs Actual')
axes[2].legend()
axes[2].set_xticks(x_pos)

plt.tight_layout()
plt.show()

---

## 🎯 Summary: What We Changed and Why

| # | Trick | Where it acts | Expected improvement |
|---|-------|--------------|---------------------|
| 🔧 1 | **Log-transform target** | Preprocessing | Lower MAPE, more uniform errors across price ranges |
| 🔧 2 | **Feature engineering** | Preprocessing | Lower MAE/RMSE, captures non-linear relationships |
| 🔧 3 | **Mini-batch DataLoader** | Training loop | Better generalization, more updates per epoch |
| 🔧 4 | **LR Scheduler** | Training loop | Fine-tuned convergence, lower final loss |
| 🔧 5 | **Kaiming initialization** | Model definition | Faster initial convergence, fewer dead neurons |

### Key takeaways for students

1. **Data preparation often matters more than architecture** — Tricks 1 and 2 (log-transform and feature engineering) are likely the biggest contributors to improvement.
2. **Training dynamics matter** — Tricks 3 and 4 (DataLoader and scheduler) don't change *what* the model learns, but *how efficiently* it learns.
3. **Small details add up** — Trick 5 (Kaiming init) is a one-liner, but combined with the others, it contributes to a cleaner training trajectory.
4. **Always compare** — Without the base notebook as a reference, we wouldn't know which tricks actually help. Controlled experiments are essential.